# Notebook 16: Multi-Stock AFML Pipeline Overview

End-to-end summary of the 10-stock leakage-resistant AFML pipeline:

| Phase | Description | Key Artifact |
|-------|-------------|-------------|
| 3 | Data acquisition (2005-2025, 10 stocks) | `panel_ohlcv.parquet` |
| 4 | Per-stock AFML pipeline (CUSUM → labels → weights → features) | `per_stock/*.parquet` |
| 5 | 101 alpha engine + diagnostics + selection | `panel_alpha_features_pruned.parquet` |
| 6 | Leakage validation (34 checks) | `leakage_audit.parquet` |
| 7 | MultiAssetPurgedKFold CV baseline | `cv_baseline_multistock.parquet` |
| 8 | Sample weight investigation + clipping | `pooled_modelling.parquet` (updated) |

**Universe**: AAPL, AMZN, BAC, GOOGL, JNJ, JPM, MSFT, NVDA, UNH, XOM  
**Period**: 2005-01-03 to 2025-04-30  
**Events**: 2,071 CUSUM events with full features (50 total: 17 TS + 33 alpha)

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('../..'))

with open('../../configs/universe.json') as f:
    UNI = json.load(f)
with open('../../configs/selected_alphas.json') as f:
    ALPHA_CFG = json.load(f)

TICKERS = UNI['tickers']
print(f'Universe : {TICKERS}')
print(f'Period   : {UNI["common_start"]} -> {UNI["common_end"]}')
print(f'Alphas   : {ALPHA_CFG["n_alphas"]} selected from 101')

## 1. Panel summary

In [ ]:
panel = pd.read_parquet('../../data/processed/panel_ohlcv.parquet')
print(f'OHLCV panel  : {panel.shape}')
print(f'Date range   : {panel.index.get_level_values("Date").min().date()} -> '
      f'{panel.index.get_level_values("Date").max().date()}')
print(f'Tickers      : {sorted(panel.index.get_level_values("ticker").unique().tolist())}')
print(f'NaN count    : {panel.isnull().sum().sum()}')

# Per-ticker row counts
rows_per_ticker = panel.groupby(level='ticker').size()
print('\nRows per ticker:')
print(rows_per_ticker.to_string())

## 2. Per-stock pipeline summary (CUSUM events → labels)

In [ ]:
per_stock_dir = '../../data/processed/per_stock'

summary_rows = []
for ticker in TICKERS:
    lbl_path = os.path.join(per_stock_dir, f'{ticker}_labels.parquet')
    feat_path = os.path.join(per_stock_dir, f'{ticker}_ts_features.parquet')
    if not os.path.exists(lbl_path):
        continue
    lbl  = pd.read_parquet(lbl_path)
    feat = pd.read_parquet(feat_path)
    n_pos = int((lbl['bin'] == 1).sum())
    n_neg = int((lbl['bin'] == -1).sum())
    summary_rows.append({
        'ticker':   ticker,
        'n_labels': len(lbl),
        'n_pos':    n_pos,
        'n_neg':    n_neg,
        'bal_pct':  round(n_pos / max(len(lbl), 1) * 100, 1),
        'n_features': len(feat.columns),
        'date_min': lbl.index.min().date(),
        'date_max': lbl.index.max().date(),
    })

summary = pd.DataFrame(summary_rows).set_index('ticker')
print(summary.to_string())
print(f'\nTotal labels: {summary["n_labels"].sum()}')

## 3. Pooled modelling dataset overview

In [ ]:
modelling = pd.read_parquet('../../data/processed/pooled_modelling.parquet')

meta_cols  = {'label', 't1', 'weight', 'ticker'}
ts_cols    = [c for c in modelling.columns if c not in meta_cols and not c.startswith('alpha')]
alpha_cols = [c for c in modelling.columns if c.startswith('alpha')]

print(f'Modelling dataset: {modelling.shape}')
print(f'  TS features    : {len(ts_cols)}')
print(f'  Alpha features : {len(alpha_cols)}')
print(f'  Total features : {len(ts_cols) + len(alpha_cols)}')
print(f'  Date range     : {modelling.index.min().date()} -> {modelling.index.max().date()}')
print(f'  Tickers        : {sorted(modelling["ticker"].unique())}')
print(f'  Labels: +1={int((modelling["label"]==1).sum())}  -1={int((modelling["label"]==-1).sum())}')
print(f'  Weight max     : {modelling["weight"].max():.4f}  (clipped at p99)')

# Per-ticker event count
by_ticker = modelling.groupby('ticker').agg(
    n_events=('label', 'count'),
    n_pos=('label', lambda x: (x==1).sum()),
    n_neg=('label', lambda x: (x==-1).sum()),
)
by_ticker['pos_pct'] = (by_ticker['n_pos'] / by_ticker['n_events'] * 100).round(1)
print('\nPer-ticker breakdown:')
print(by_ticker.to_string())

## 4. CV baseline results

In [ ]:
cv_df = pd.read_parquet('../../data/processed/cv_baseline_multistock.parquet')
print('MultiAssetPurgedKFold 5-fold CV results:')
print(cv_df[['model','fold','n_train','n_test','accuracy','majority_base','beats_majority']].to_string(index=False))

for model in cv_df['model'].unique():
    sub = cv_df[cv_df['model'] == model]
    print(f'\n{model}: mean acc={sub["accuracy"].mean():.4f}  '
          f'vs majority={sub["majority_base"].mean():.4f}')

## 5. Leakage audit summary

In [ ]:
audit = pd.read_parquet('../../data/processed/leakage_audit.parquet')
print(f'Leakage audit: {len(audit)} checks')
n_pass = int((audit['status'] == 'PASS').sum())
n_warn = int((audit['status'] == 'WARN').sum())
n_fail = int((audit['status'] == 'FAIL').sum())
print(f'  PASS: {n_pass}  WARN: {n_warn}  FAIL: {n_fail}')
print()
if n_warn > 0:
    print('WARNINGS:')
    print(audit[audit['status']=='WARN'][['group','name','detail']].to_string(index=False))
if n_fail > 0:
    print('FAILURES:')
    print(audit[audit['status']=='FAIL'][['group','name','detail']].to_string(index=False))

## 6. Price history plot (adjusted close, all 10 stocks)

In [ ]:
close_wide = panel['AdjClose'].unstack(level='ticker')
normalized = close_wide.div(close_wide.iloc[0])  # index to 1.0 at start

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab10.colors
for i, ticker in enumerate(sorted(close_wide.columns)):
    ax.plot(normalized.index, normalized[ticker], label=ticker,
            color=colors[i], linewidth=1.2, alpha=0.85)

ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price (2005-01-03 = 1.0)')
ax.set_title('10-Stock Universe — Adjusted Close (Normalized, 2005-2025)')
ax.legend(ncol=5, loc='upper left', fontsize=9)
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('../../reports/figures/pooled/9_price_history.png', dpi=100)
plt.show()